In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
CREATE WIDGET TEXT CATALOG default "adbdep_0013";
CREATE WIDGET TEXT STORAGE default "adlsproject13dev2";

In [0]:
CREATE TABLE IF NOT EXISTS IDENTIFIER(:CATALOG || '.control.etl_run_log') (
    run_id STRING,
    pipeline_name STRING,
    flow_name STRING,
    layer STRING,
    object_name string,
    start_ts TIMESTAMP,
    end_ts TIMESTAMP,
    duration_sec BIGINT,
    status STRING,
    rows_written BIGINT,
    created_at TIMESTAMP,
    error_message STRING,
    error_detail STRING
)
USING DELTA;

In [0]:
MERGE INTO IDENTIFIER(:CATALOG || '.control.etl_run_log') AS tgt

USING (

    WITH flow_events AS (

        SELECT
            origin.update_id AS run_id,
            origin.pipeline_name AS pipeline_name,
            origin.flow_name AS flow_name,

            timestamp,

            details:flow_progress.status::STRING AS status,

            TRY_CAST(
                details:flow_progress.metrics.num_output_rows
                AS BIGINT
            ) AS rows_written

        FROM event_log(TABLE(control.processed_files))

        WHERE event_type = 'flow_progress'
          AND origin.flow_name IS NOT NULL
          AND origin.flow_name NOT LIKE 'pipelines.%'
    ),

    flow_metrics AS (

        SELECT
            run_id,
            pipeline_name,
            flow_name,

            MIN(timestamp) AS start_ts,
            MAX(timestamp) AS end_ts,
            MAX(rows_written) AS rows_written

        FROM flow_events

        GROUP BY
            run_id,
            pipeline_name,
            flow_name
    ),

    flow_status AS (

        SELECT
            run_id,
            flow_name,
            status

        FROM (
            SELECT
                run_id,
                flow_name,
                status,

                ROW_NUMBER() OVER (
                    PARTITION BY run_id, flow_name
                    ORDER BY timestamp DESC
                ) AS rn

            FROM flow_events

            WHERE status IS NOT NULL
        )

        WHERE rn = 1
    ),
        error_events AS (

        SELECT
            origin.update_id AS run_id,
            origin.flow_name AS flow_name,

            timestamp AS error_ts,

            message AS error_message,

            error AS error_detail,

            ROW_NUMBER() OVER (
                PARTITION BY
                    origin.update_id,
                    origin.flow_name
                ORDER BY timestamp DESC
            ) AS rn

        FROM event_log(TABLE(control.processed_files))

        WHERE
            level = 'ERROR'
            OR error IS NOT NULL
    ),
        latest_error AS (
            SELECT
                run_id,
                flow_name,
                error_message,
                error_detail

            FROM error_events

            WHERE rn = 1
    )

    SELECT
        m.run_id,
        m.pipeline_name,
        m.flow_name,

        CASE
            WHEN m.flow_name LIKE '%.ly_bronze.%' THEN 'BRONZE'
            WHEN m.flow_name LIKE '%.ly_silver.%' THEN 'SILVER'
            WHEN m.flow_name LIKE '%.ly_gold.%' THEN 'GOLD'
            WHEN m.flow_name LIKE '%.control.%' THEN 'CONTROL'
            ELSE 'OTHER'
        END AS layer,

        element_at(
            split(m.flow_name, '\\.'),
            -1
        ) AS object_name,

        m.start_ts,
        m.end_ts,

        TIMESTAMPDIFF(
            SECOND,
            m.start_ts,
            m.end_ts
        ) AS duration_sec,

        s.status,
        m.rows_written,

        current_timestamp() AS created_at,
        e.error_message,
        e.error_detail

    FROM flow_metrics m

    LEFT JOIN flow_status s
        ON m.run_id = s.run_id
        AND m.flow_name = s.flow_name

    LEFT JOIN latest_error e
    ON m.run_id = e.run_id
    AND m.flow_name = e.flow_name

    WHERE m.pipeline_name = 'ingest_y_transform'

) AS src

ON  tgt.run_id = src.run_id
AND tgt.flow_name = src.flow_name

WHEN MATCHED THEN UPDATE SET
    tgt.pipeline_name = src.pipeline_name,
    tgt.layer = src.layer,
    tgt.object_name = src.object_name,
    tgt.start_ts = src.start_ts,
    tgt.end_ts = src.end_ts,
    tgt.duration_sec = src.duration_sec,
    tgt.status = src.status,
    tgt.rows_written = src.rows_written,
    tgt.error_message = src.error_message,
    tgt.error_detail = src.error_detail
WHEN NOT MATCHED THEN INSERT (
    run_id,
    pipeline_name,
    flow_name,
    layer,
    object_name,
    start_ts,
    end_ts,
    duration_sec,
    status,
    rows_written,
    created_at,
    error_message,
    error_detail
)
VALUES (
    src.run_id,
    src.pipeline_name,
    src.flow_name,
    src.layer,
    src.object_name,
    src.start_ts,
    src.end_ts,
    src.duration_sec,
    src.status,
    src.rows_written,
    src.created_at,
    src.error_message,
    src.error_detail
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
90,0,0,90


#### ADICION ERORRES

In [0]:
SELECT * FROM adbdep_0013.control.etl_run_log ;


run_id,pipeline_name,flow_name,layer,object_name,start_ts,end_ts,duration_sec,status,rows_written,created_at,error_message,error_detail
555ab7b3-f890-4937-a871-8b735c5dc024,ingest_y_transform,adbdep_0013.ly_silver.transactions,SILVER,transactions,2026-08-17T14:56:23.078Z,2026-08-17T14:56:23.078Z,0,FAILED,null,2026-08-18T19:29:10.774Z,Failed to resolve flow due to upstream failure: 'adbdep_0013.ly_silver.transactions'.,"{true, [{pyspark.errors.exceptions.captured.AnalysisException, Traceback (most recent call last): pyspark.errors.exceptions.captured.AnalysisException: Failed to read dataset 'txn_valid'. Dataset is defined in the pipeline but could not be resolved., null}]}"
8e24788b-4ed3-45ab-8454-f552b8fa1814,ingest_y_transform,adbdep_0013.ly_bronze.txn_rw,BRONZE,txn_rw,2026-08-15T17:19:05.067Z,2026-08-15T17:19:22.401Z,17,COMPLETED,null,2026-08-18T19:29:10.774Z,null,null
5a9df514-fdbc-4f50-89b5-181a44d03c22,ingest_y_transform,adbdep_0013.ly_silver.quarantine_customer,SILVER,quarantine_customer,2026-08-15T16:47:44.780Z,2026-08-15T16:48:01.809Z,17,COMPLETED,0,2026-08-18T19:29:10.774Z,null,null
62f432c0-ae08-4623-b69a-804a5c1504be,ingest_y_transform,adbdep_0013.ly_bronze.account_rw,BRONZE,account_rw,2026-08-15T16:44:52.430Z,2026-08-15T16:44:58.074Z,5,COMPLETED,null,2026-08-18T19:29:10.774Z,null,null
f263c3fb-cb1d-4866-9af1-731307ff7caa,ingest_y_transform,adbdep_0013.ly_silver.quarantine_txn,SILVER,quarantine_txn,2026-08-17T14:56:05.019Z,2026-08-17T14:56:05.019Z,0,FAILED,null,2026-08-18T19:29:10.774Z,Failed to resolve flow due to upstream failure: 'adbdep_0013.ly_silver.quarantine_txn'.,"{true, [{pyspark.errors.exceptions.captured.AnalysisException, Traceback (most recent call last): File ""REDACTED_LOCAL_PART@gmail.com/pipeline_process/transformations/silver_transformation_txn.py"", cell 1, line 168, in quarantine_txn pyspark.errors.exceptions.captured.AnalysisException: Failed to read dataset 'txn_dq'. Dataset is defined in the pipeline but could not be resolved., null}]}"
c3a07bd0-e8b9-4dd8-96ee-68b7544c3c2d,ingest_y_transform,adbdep_0013.ly_silver.quarantine_customer,SILVER,quarantine_customer,2026-08-15T17:43:43.225Z,2026-08-15T17:44:11.974Z,28,COMPLETED,0,2026-08-18T19:29:10.774Z,null,null
c3a07bd0-e8b9-4dd8-96ee-68b7544c3c2d,ingest_y_transform,adbdep_0013.ly_silver.dim_customer,SILVER,dim_customer,2026-08-15T17:43:43.220Z,2026-08-15T17:44:11.967Z,28,COMPLETED,null,2026-08-18T19:29:10.774Z,null,null
9b6d0f99-25e4-45ab-9b69-e6ee7badc641,ingest_y_transform,account_dq,OTHER,account_dq,2026-08-15T17:40:39.749Z,2026-08-15T17:40:39.749Z,0,FAILED,null,2026-08-18T19:29:10.774Z,Failed to resolve flow: 'account_dq'.,"{true, [{pyspark.errors.exceptions.captured.AnalysisException, Traceback (most recent call last): File ""REDACTED_LOCAL_PART@gmail.com/pipeline_process/transformations/silver_dimensions_account.py"", cell 1, line 50, in account_dq pyspark.errors.exceptions.captured.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `c`.`customer_id` cannot be resolved. Did you mean one of the following? [`a`.`customer_id`, `c`.`cust_customer_id`, `a`.`account_id`, `a`.`status`, `a`.`updated_at`]. SQLSTATE: 42703; 'Join LeftOuter, '`=`('c.customer_id, 'a.cust_customer_id) :- ~SubqueryAlias a : +- ~Project [account_id#12119, customer_id#12120, status#12121, to_date(opened_date#12122, None, Some(Etc/UTC), false) AS opened_date#12129, updated_at#12128, _rescued_data#12124, source_file#12125, file_mod_time#12126, ingestion_ts#12127] : +- ~Project [account_id#12119, customer_id#12120, status#12121, opened_date#12122, to_timestamp(updated_at#12123, None, TimestampType, Some(Etc/UTC), false) AS updated_at#12128, _rescued_data#12124, source_file#12125, file_mod_time#12126, ingestion_ts#12127] : +- ~SubqueryAlias adbdep_0013.ly_bronze.account_rw : +- ~StreamingRelationV2 org.apache.spark.sql.execution.streaming.runtime.MemoryStreamTableProvider$@f6ba49c, memory, org.apache.spark.sql.execution